In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from pathlib import Path

from hll_euler_eq_1d_solver import HLL_Euler_Eq_Solver


# ============================================================
# Parameters
# ============================================================

gamma = 7 / 5

dt = 1e-5
N = 500
dx = 0.01


x = (np.arange(N) + 0.5) * dx

t = 0.0
t_end = 1.0

# Ambient medium
rho0 = 1.0
p_ambient = 1e-6

# Initial hot region
r_explosion = 0.1
p_explosion = 1000.0

# Times at which we want snapshots
snapshot_times = [0.02, 0.05, 0.1, 0.2, 0.5, 1.0]

# Output directory
output_dir = Path("sedov_snapshots")
output_dir.mkdir(exist_ok=True)


# ============================================================
# Solver
# ============================================================

solver = HLL_Euler_Eq_Solver(
    dt=dt,
    dx=dx,
    gamma=gamma
)


# ============================================================
# Initial condition
# ============================================================

U = np.zeros((N, 3))

for i in range(N):

    rho = rho0
    v = 0.0

    if x[i] <= r_explosion:
        p = p_explosion
    else:
        p = p_ambient

    momentum = rho * v

    E = (
        p / (gamma - 1)
        + 0.5 * rho * v**2
    )

    U[i] = [rho, momentum, E]


# ============================================================
# Exact Sedov-Taylor similarity solution
# ============================================================

def euler_eqs(s, z):

    w, lnG = z

    u = np.exp(w)

    V = (u + 1) / gamma

    A = (
        2
        - (3 + gamma) * V
        + 2 * gamma * V**2
    ) / (
        (V - 1) * V * u
    )

    B = (
        5
        - (3 * gamma - 1) * V
    ) / (
        (gamma - 1) - (1 - V) * A
    )

    return [
        gamma * B / u,
        (
            ((5 - 2 * V) / (1 - V))
            + A * B
        ) / (gamma - 1)
    ]


w0 = np.log(
    (2 * gamma / (gamma + 1)) - 1
)

G0 = np.log(
    (gamma + 1) / (gamma - 1)
)

sol = solve_ivp(
    euler_eqs,
    [0, -10],
    [w0, G0],
    rtol=1e-12,
    atol=1e-14,
    method="Radau"
)


s = sol.t[::-1]

xi_exact = np.exp(s)

V_exact = (
    np.exp(sol.y[0][::-1]) + 1
) / gamma

G_exact = np.exp(
    sol.y[1][::-1]
)

Z_exact = (
    gamma
    * (gamma - 1)
    * (1 - V_exact)
    * V_exact**2
    / (
        2 * np.exp(sol.y[0][::-1])
    )
)

v_exact = (
    V_exact
    * xi_exact
    * (gamma + 1)
    / 2
)

rho_exact = (
    G_exact
    * (gamma - 1)
    / (gamma + 1)
)

p_exact = (
    Z_exact
    * G_exact
    * xi_exact**2
    * (gamma + 1)
    / (2 * gamma)
)

T_exact = (
    xi_exact**2
    * Z_exact
    * (gamma + 1)**2
    / (
        2 * gamma * (gamma - 1)
    )
)


# ============================================================
# Helper functions
# ============================================================

def primitive_variables(U):

    rho = U[:, 0]
    momentum = U[:, 1]
    E = U[:, 2]

    v = momentum / rho

    p = (
        gamma - 1
    ) * (
        E
        - momentum**2 / (2 * rho)
    )

    return rho, v, p


def find_shock_radius(U):

    rho = U[:, 0]

    density_gradient = np.abs(
        np.gradient(rho, x)
    )

    shock_index = np.argmax(
        density_gradient
    )

    return x[shock_index]


def get_normalized_profiles(U, t, R):

    rho, v, p = primitive_variables(U)

    xi = x / R

    # Sedov-Taylor shock velocity
    D = (2 * R / (5 * t))

    # Gas velocity immediately behind the shock front
    v_shock = 2*D/(gamma+1)

    # Strong-shock post-shock density
    rho_shock = (
        rho0
        * (gamma + 1)
        / (gamma - 1)
    )

    # Strong-shock post-shock pressure
    p_shock = (
        2
        * rho0
        * D**2
        / (gamma + 1)
    )

    # Since T ∝ p/rho
    T_shock = p_shock / rho_shock
    
    v_norm = v / v_shock
    rho_norm = rho / rho_shock
    p_norm = p / p_shock
    T_norm = (
        (p / rho)
        / T_shock
    )

    return (
        xi,
        v_norm,
        rho_norm,
        p_norm,
        T_norm
    )


def save_snapshot(U, t, R, snapshot_time):

    # --------------------------------------------------------
    # Physical variables
    # --------------------------------------------------------

    rho, v, p = primitive_variables(U)

    # --------------------------------------------------------
    # Normalized variables
    # --------------------------------------------------------

    (
        xi_num,
        v_norm,
        rho_norm,
        p_norm,
        T_norm
    ) = get_normalized_profiles(
        U, t, R
    )

    # --------------------------------------------------------
    # Create comparison figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(10, 8)
    )

    # Velocity
    ax = axes[0, 0]

    ax.plot(
        xi_exact,
        v_exact,
        "--",
        label="Sedov-Taylor"
    )

    ax.plot(
        xi_num,
        v_norm,
        label="Numerical"
    )

    ax.set_xlabel(r"$\xi$")
    ax.set_ylabel(r"$v/v_s$")
    ax.set_title("Velocity")
    ax.set_xlim(0, 1)
    ax.grid()
    ax.legend()

    # Density
    ax = axes[0, 1]

    ax.plot(
        xi_exact,
        rho_exact,
        "--",
        label="Sedov-Taylor"
    )

    ax.plot(
        xi_num,
        rho_norm,
        label="Numerical"
    )

    ax.set_xlabel(r"$\xi$")
    ax.set_ylabel(r"$\rho/\rho_s$")
    ax.set_title("Density")
    ax.set_xlim(0, 1)
    ax.grid()
    ax.legend()

    # Pressure
    ax = axes[1, 0]

    ax.plot(
        xi_exact,
        p_exact,
        "--",
        label="Sedov-Taylor"
    )

    ax.plot(
        xi_num,
        p_norm,
        label="Numerical"
    )

    ax.set_xlabel(r"$\xi$")
    ax.set_ylabel(r"$p/p_s$")
    ax.set_title("Pressure")
    ax.set_xlim(0, 1)
    ax.grid()
    ax.legend()

    # Temperature
    ax = axes[1, 1]

    ax.plot(
        xi_exact,
        T_exact,
        "--",
        label="Sedov-Taylor"
    )

    ax.plot(
        xi_num,
        T_norm,
        label="Numerical"
    )

    ax.set_xlabel(r"$\xi$")
    ax.set_ylabel(r"$T/T_s$")
    ax.set_title("Temperature")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1000)
    ax.grid()
    ax.legend()

    fig.suptitle(
        f"Spherical blast wave: "
        f"t = {t:.3f}, R = {R:.4f}"
    )

    fig.tight_layout()

    fig.savefig(
        output_dir
        / f"snapshot_{snapshot_time:.3f}.png",
        dpi=150
    )

    plt.close(fig)

    print(
        f"Snapshot saved: "
        f"t = {t:.5f}, "
        f"R = {R:.5f}"
    )



# ============================================================
# Main simulation
# ============================================================

next_snapshot = 0

while t < t_end:

    U = solver.time_step_spherical(U)

    t += dt

    # --------------------------------------------------------
    # Calculate shock radius
    # --------------------------------------------------------

    R = find_shock_radius(U)

    # --------------------------------------------------------
    # Save requested snapshot
    # --------------------------------------------------------

    if (
        next_snapshot < len(snapshot_times)
        and t >= snapshot_times[next_snapshot]
    ):

        save_snapshot(
            U,
            t,
            R,
            snapshot_times[next_snapshot]
        )

        next_snapshot += 1

    

Snapshot saved: t = 0.02001, R = 0.36500
Snapshot saved: t = 0.05000, R = 0.52500
Snapshot saved: t = 0.10001, R = 0.68500
Snapshot saved: t = 0.20000, R = 0.90500
Snapshot saved: t = 0.50000, R = 1.29500
Snapshot saved: t = 1.00001, R = 1.71500
